# MeanFlow 论文详细学习笔记
## Mean Flows for One-step Generative Modeling

**论文：** *Mean Flows for One-step Generative Modeling*  
**论文链接：** https://arxiv.org/abs/2505.13447  
**原文 PDF：** https://arxiv.org/pdf/2505.13447

这次将全文统一改为我们更熟悉的 Flow Matching 时间约定：

$$
\boxed{
x_0\sim p_0=\mathcal N(0,I)
\quad\longrightarrow\quad
x_1\sim p_1=p_{\mathrm{data}}.
}
$$

也就是：

- $x_0$：初始高斯噪声；
- $x_1$：真实数据；
- $x_t$：时间 $t$ 的中间状态；
- $t=0$：噪声端；
- $t=1$：数据端；
- 生成方向：$0\rightarrow1$。

**特别注意：原论文采用的是相反的时间方向。** 因此，下面不仅会替换符号，还会同步调整时间区间、积分方向、MeanFlow Identity 中的符号、JVP 方向和采样公式，保证整套推导在新的约定下严格一致。

全文按照论文的理论与实验顺序展开，并在最后补充高级数学知识。

---

# 第一部分：全文核心结论

## 1. MeanFlow 解决了什么问题？

标准 Flow Matching 学习的是瞬时速度场：

$$
\boxed{
v(x_t,t)=\frac{dx_t}{dt}.
}
\tag{1.1}
$$

给定初始噪声：

$$
x_0\sim\mathcal N(0,I),
$$

需要通过 ODE 积分得到生成数据：

$$
\boxed{
x_1=x_0+\int_0^1v(x_\tau,\tau)\,d\tau.
}
\tag{1.2}
$$

但神经网络：

$$
v_\theta(x_t,t)
$$

只能预测某个时刻的瞬时速度。

如果直接使用：

$$
x_1\approx x_0+v_\theta(x_0,0),
$$

相当于使用初始时刻的速度近似整个区间的速度积分。

当速度场随时间和位置变化时，这种近似通常不准确。

MeanFlow 提出：

> 不再直接预测瞬时速度，而是预测一段时间内的平均速度。

对于：

$$
0\leq s<t\leq1,
$$

定义：

$$
\boxed{
u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
}
\tag{1.3}
$$

其中：

- $s$：起始时间；
- $t$：目标时间；
- $x_s$：起始状态；
- $v$：瞬时速度场；
- $u$：区间 $[s,t]$ 上的平均速度场。

由 ODE 的积分形式：

$$
x_t-x_s=\int_s^t v(x_\tau,\tau)\,d\tau,
$$

可得：

$$
\boxed{
x_t=x_s+(t-s)u(x_s,s,t).
}
\tag{1.4}
$$

特别地，当：

$$
s=0,\qquad t=1,
$$

有：

$$
\boxed{
x_1=x_0+u(x_0,0,1).
}
\tag{1.5}
$$

因此，只要神经网络准确预测：

$$
u_\theta(x_0,0,1),
$$

就能够通过一次前向传播生成数据。

这就是 MeanFlow 实现一步生成的基本思想。

---

## 2. 全文最重要的理论公式

根据平均速度定义：

$$
(t-s)u(x_s,s,t)
=
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

保持目标时间 $t$ 固定，对起始时间 $s$ 求导，可以得到：

$$
\boxed{
u(x_s,s,t)
=
v(x_s,s)
+
(t-s)\frac{d}{ds}u(x_s,s,t).
}
\tag{2.1}
$$

这就是在**噪声 $0\rightarrow$ 数据 $1$** 的时间约定下，MeanFlow Identity 的形式。

注意右边是加号，而不是原论文时间方向下的减号。

进一步利用链式法则：

$$
\boxed{
\frac{d}{ds}u(x_s,s,t)
=
\partial_su(x_s,s,t)
+
J_u(x_s,s,t)v(x_s,s).
}
\tag{2.2}
$$

因此：

$$
\boxed{
u
=
v+(t-s)
\left(
\partial_su+J_uv
\right).
}
\tag{2.3}
$$

这是本文最核心的训练恒等式。

它表明：

**平均速度可以通过当前瞬时速度，以及平均速度沿 ODE 轨迹的方向导数来表示。**

所以，训练时不必先计算完整的 ODE 积分。

---

## 3. 核心训练目标

定义真实数据：

$$
x_1\sim p_{\mathrm{data}},
$$

初始噪声：

$$
x_0\sim\mathcal N(0,I).
$$

采用线性插值：

$$
\boxed{
x_s=(1-s)x_0+sx_1.
}
\tag{3.1}
$$

条件速度：

$$
\boxed{
V=x_1-x_0.
}
\tag{3.2}
$$

利用网络：

$$
u_\theta(x_s,s,t)
$$

预测平均速度。

构造 JVP：

$$
\boxed{
D_su_\theta
=
\operatorname{JVP}
\left(
u_\theta,
(x_s,s,t),
(V,1,0)
\right).
}
\tag{3.3}
$$

其中：

$$
(V,1,0)
$$

分别表示：

- 空间状态沿 $V$ 变化；
- 起始时间 $s$ 以单位速度变化；
- 目标时间 $t$ 保持不变。

因此：

$$
D_su_\theta
=
J_{u_\theta}V+\partial_su_\theta.
$$

根据 MeanFlow Identity：

$$
\boxed{
u_{\mathrm{target}}
=
V+(t-s)D_su_\theta.
}
\tag{3.4}
$$

训练损失：

$$
\boxed{
\mathcal L_{\mathrm{MF}}
=
\mathbb E
\left[
\left\|
u_\theta(x_s,s,t)
-
\operatorname{sg}(u_{\mathrm{target}})
\right\|^2
\right].
}
\tag{3.5}
$$

其中：

$$
\operatorname{sg}
$$

表示停止梯度。

这就是在新时间方向下的 MeanFlow 训练目标。

---

## 4. 全文的核心结论

**结论一：MeanFlow 学习的是有限时间区间的平均速度，而不是瞬时速度。**

$$
u(x_s,s,t)
=
\frac{x_t-x_s}{t-s}.
$$

**结论二：平均速度能够由一个严格的微分恒等式定义。**

$$
u=v+(t-s)(\partial_su+J_uv).
$$

**结论三：可以直接利用条件 Flow Matching 的速度标签训练平均速度模型。**

$$
V=x_1-x_0.
$$

不需要先求解完整 ODE，也不需要预训练教师模型。

**结论四：MeanFlow 支持一步生成。**

$$
\boxed{
\hat x_1=x_0+u_\theta(x_0,0,1).
}
$$

**结论五：MeanFlow 可以在训练时纳入 CFG，并保持一次网络推理生成。**

**结论六：论文在 ImageNet 256×256 上实现了 FID 3.43、NFE 1 的生成效果。**

---

# 第二部分：Background — Flow Matching

## 5. 概率插值路径

设：

$$
x_0\sim p_0,
$$

$$
x_1\sim p_1.
$$

其中：

$$
p_0=\mathcal N(0,I),
\qquad
p_1=p_{\mathrm{data}}.
$$

定义一般形式的概率插值路径：

$$
\boxed{
x_t=a_tx_0+b_tx_1.
}
\tag{5.1}
$$

要求：

$$
a_0=1,\qquad b_0=0,
$$

$$
a_1=0,\qquad b_1=1.
$$

因此：

$$
x_{t=0}=x_0,
$$

$$
x_{t=1}=x_1.
$$

论文采用线性插值。

在我们的时间约定下：

$$
a_t=1-t,
\qquad
b_t=t.
$$

因此：

$$
\boxed{
x_t=(1-t)x_0+tx_1.
}
\tag{5.2}
$$

对时间求导：

$$
\begin{aligned}
\frac{dx_t}{dt}
&=
\frac{d}{dt}
\left[
(1-t)x_0+tx_1
\right]\\
&=
-x_0+x_1.
\end{aligned}
$$

所以条件速度为：

$$
\boxed{
V=x_1-x_0.
}
\tag{5.3}
$$

它与时间无关。

但需要区分：这里的 $V$ 是一对样本 $(x_0,x_1)$ 对应的条件速度，不是最终生成 ODE 的边缘速度场。

---

## 6. 条件速度与边缘速度

给定：

$$
x_t=x,
$$

可能存在多个样本对：

$$
(x_0,x_1)
$$

满足：

$$
x=(1-t)x_0+tx_1.
$$

不同样本对的条件速度：

$$
V=x_1-x_0
$$

可能不同。

因此，定义边缘速度场：

$$
\boxed{
v(x,t)
=
\mathbb E[V\mid x_t=x].
}
\tag{6.1}
$$

这个速度场是确定性的。

给定：

$$
x,t,
$$

就得到唯一的速度向量：

$$
v(x,t).
$$

### 6.1 为什么 MSE 学习的是边缘速度？

假设网络预测：

$$
v_\theta(x_t,t).
$$

标准 Flow Matching 损失：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E
\left[
\|v_\theta(x_t,t)-V\|^2
\right].
}
\tag{6.2}
$$

定义：

$$
m(x,t)=\mathbb E[V\mid x_t=x].
$$

将误差拆开：

$$
v_\theta-V
=
(v_\theta-m)+(m-V).
$$

平方展开：

$$
\begin{aligned}
\|v_\theta-V\|^2
={}&
\|v_\theta-m\|^2\\
&+
\|m-V\|^2\\
&+
2(v_\theta-m)^\top(m-V).
\end{aligned}
$$

对 $x_t,t$ 条件求期望：

$$
\mathbb E[m-V\mid x_t,t]=0.
$$

因此交叉项为零：

$$
\mathbb E[
(v_\theta-m)^\top(m-V)
]=0.
$$

所以：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{FM}}
={}&
\mathbb E\|v_\theta-m\|^2\\
&+
\mathbb E\|m-V\|^2.
\end{aligned}
}
\tag{6.3}
$$

第二项不依赖网络参数。

因此：

$$
\boxed{
v_\theta^*(x,t)
=
m(x,t)
=
\mathbb E[V\mid x_t=x].
}
\tag{6.4}
$$

这说明，使用样本级条件速度：

$$
V=x_1-x_0
$$

作为监督，能够学习边缘速度场。

---

## 7. 边缘速度场对应的 ODE

标准 Flow Matching 生成过程：

$$
\boxed{
\frac{dx_t}{dt}=v(x_t,t).
}
\tag{7.1}
$$

从：

$$
x_0\sim p_0
$$

开始积分。

当：

$$
t=1,
$$

得到：

$$
x_1\sim p_1.
$$

积分形式：

$$
\boxed{
x_t=x_s+\int_s^t v(x_\tau,\tau)\,d\tau.
}
\tag{7.2}
$$

特别地：

$$
\boxed{
x_1=x_0+\int_0^1v(x_\tau,\tau)\,d\tau.
}
\tag{7.3}
$$

如果使用 Euler：

$$
x_{t+\Delta t}
\approx
x_t+\Delta t\,v(x_t,t).
$$

但这只是使用某个时刻的瞬时速度近似整个小区间的积分。

MeanFlow 的出发点就是直接学习：

$$
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

---

# 第三部分：MeanFlow Models

对应论文第 4.1 节，是全文最核心的理论部分。

## 8. Average Velocity：平均速度的定义

### 8.1 从 ODE 的积分形式出发

已知边缘 ODE：

$$
\frac{dx_t}{dt}=v(x_t,t).
$$

对于：

$$
0\leq s<t\leq1,
$$

积分得到：

$$
x_t-x_s
=
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

两边除以时间间隔：

$$
t-s,
$$

得到：

$$
\frac{x_t-x_s}{t-s}
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

定义平均速度：

$$
\boxed{
u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
}
\tag{8.1}
$$

等价地：

$$
\boxed{
u(x_s,s,t)
=
\frac{x_t-x_s}{t-s}.
}
\tag{8.2}
$$

需要强调：

**这里的 $x_t$ 是从 $x_s$ 出发，沿边缘速度场 $v$ 积分得到的终点，而不一定是与训练样本 $x_0,x_1$ 对应的条件直线插值点。**

### 8.2 平均速度的物理意义

瞬时速度：

$$
v(x_s,s)
$$

表示当前时刻的变化率。

平均速度：

$$
u(x_s,s,t)
$$

表示从时间 $s$ 到时间 $t$ 的净位移除以时间长度。

因此：

$$
\boxed{
x_t=x_s+(t-s)u(x_s,s,t).
}
\tag{8.3}
$$

例如，某个一维 ODE 的瞬时速度为：

$$
v(x_\tau,\tau)=2\tau.
$$

从 $s=0$ 到 $t=1$：

$$
x_1-x_0
=
\int_0^1 2\tau\,d\tau
=1.
$$

因此：

$$
u(x_0,0,1)=1.
$$

但初始瞬时速度：

$$
v(x_0,0)=0.
$$

如果只使用初始瞬时速度进行一步 Euler：

$$
\hat x_1=x_0+v(x_0,0)=x_0,
$$

结果显然不正确。

而使用平均速度：

$$
\hat x_1=x_0+u(x_0,0,1)=x_0+1,
$$

可以精确得到终点。

这说明平均速度与瞬时速度是不同的数学对象。

---

## 9. 平均速度场与 ODE 流映射

为了严格定义：

$$
u(x_s,s,t),
$$

需要引入 ODE 流映射。

定义：

$$
\boxed{
\Phi_{t\leftarrow s}(x)
}
\tag{9.1}
$$

表示从时间 $s$ 的状态 $x$ 出发，沿 ODE 积分到时间 $t$ 得到的状态。

因此：

$$
\boxed{
x_t=\Phi_{t\leftarrow s}(x_s).
}
\tag{9.2}
$$

根据平均速度定义：

$$
u(x_s,s,t)
=
\frac{x_t-x_s}{t-s}.
$$

代入流映射：

$$
\boxed{
u(x,s,t)
=
\frac{
\Phi_{t\leftarrow s}(x)-x
}{
t-s
}.
}
\tag{9.3}
$$

因此：

$$
\boxed{
\Phi_{t\leftarrow s}(x)
=
x+(t-s)u(x,s,t).
}
\tag{9.4}
$$

这表明平均速度场实际上编码了完整的有限时间 ODE 流映射。

如果知道：

$$
u(x,s,t),
$$

就能直接计算从 $s$ 到 $t$ 的终点。

不需要再进行数值积分。

### 9.1 平均速度为什么需要三个输入？

平均速度场：

$$
u(x,s,t)
$$

包含三个自变量。

第一个：

$$
x
$$

表示当前状态。

第二个：

$$
s
$$

表示当前状态对应的起始时间。

第三个：

$$
t
$$

表示希望到达的目标时间。

同一个空间位置：

$$
x
$$

即使目标时间相同，只要起始时间不同，后续 ODE 轨迹也可能不同。

同样，即使起始时间相同，只要目标时间不同，平均速度也可能不同。

因此，一般需要：

$$
\boxed{
u:\mathbb R^d\times
\{(s,t):0\leq s\leq t\leq1\}
\rightarrow\mathbb R^d.
}
\tag{9.5}
$$

---

## 10. 平均速度的两个基本性质

### 10.1 性质一：零时间间隔退化为瞬时速度

根据定义：

$$
u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

令：

$$
t\rightarrow s^+.
$$

如果：

$$
v(x_\tau,\tau)
$$

沿轨迹连续，那么：

$$
\lim_{t\to s^+}
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau
=
v(x_s,s).
$$

因此：

$$
\boxed{
u(x_s,s,s)=v(x_s,s).
}
\tag{10.1}
$$

严格来说，左边通过连续延拓定义。

这个关系是 MeanFlow 的重要边界条件。

当：

$$
s=t,
$$

平均速度等于瞬时速度。

### 10.2 性质二：平均速度满足区间组合关系

考虑：

$$
0\leq s<m<t\leq1.
$$

根据积分可加性：

$$
\int_s^t v(x_\tau,\tau)\,d\tau
=
\int_s^m v(x_\tau,\tau)\,d\tau
+
\int_m^t v(x_\tau,\tau)\,d\tau.
$$

分别利用平均速度定义：

$$
\int_s^m v\,d\tau
=
(m-s)u(x_s,s,m),
$$

$$
\int_m^t v\,d\tau
=
(t-m)u(x_m,m,t).
$$

而：

$$
\int_s^t v\,d\tau
=
(t-s)u(x_s,s,t).
$$

因此：

$$
\boxed{
\begin{aligned}
(t-s)u(x_s,s,t)
={}&
(m-s)u(x_s,s,m)\\
&+
(t-m)u(x_m,m,t).
\end{aligned}
}
\tag{10.2}
$$

其中中间状态：

$$
\boxed{
x_m=x_s+(m-s)u(x_s,s,m).
}
\tag{10.3}
$$

这意味着：

$$
\boxed{
\text{一次完整区间更新}
=
\text{多次子区间更新}.
}
\tag{10.4}
$$

对于精确平均速度场，这个性质严格成立。

---

## 11. MeanFlow Identity：核心恒等式的完整推导

这是全文最重要的推导。

### 11.1 从平均速度定义出发

定义：

$$
u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

两边乘以：

$$
t-s,
$$

得到：

$$
\boxed{
(t-s)u(x_s,s,t)
=
\int_s^t v(x_\tau,\tau)\,d\tau.
}
\tag{11.1}
$$

接下来需要对这个公式求导。

由于我们采用：

$$
0\rightarrow1
$$

的生成方向，并且网络输入的是起始状态：

$$
x_s,
$$

所以选择：

**固定目标时间 $t$，对起始时间 $s$ 求导。**

### 11.2 左边使用乘积求导

定义：

$$
F(s)=(t-s)u(x_s,s,t).
$$

注意：

$$
x_s=x(s).
$$

因此：

$$
u(x_s,s,t)
$$

不仅显式依赖 $s$，还通过 $x_s$ 隐式依赖 $s$。

根据乘积法则：

$$
\begin{aligned}
\frac{dF}{ds}
={}&
\frac{d(t-s)}{ds}u(x_s,s,t)\\
&+
(t-s)\frac{d}{ds}u(x_s,s,t).
\end{aligned}
$$

因为：

$$
\frac{d(t-s)}{ds}=-1,
$$

所以：

$$
\boxed{
\frac{dF}{ds}
=
-u(x_s,s,t)
+
(t-s)\frac{d}{ds}u(x_s,s,t).
}
\tag{11.2}
$$

### 11.3 右边使用 Leibniz 积分求导法则

定义：

$$
G(s)=\int_s^t v(x_\tau,\tau)\,d\tau.
$$

由于：

$$
t
$$

固定，而积分下限为：

$$
s,
$$

根据微积分基本定理：

$$
\boxed{
\frac{dG}{ds}
=
-v(x_s,s).
}
\tag{11.3}
$$

注意负号来自积分下限的变化。

### 11.4 两边导数相等

因为：

$$
F(s)=G(s),
$$

所以：

$$
\frac{dF}{ds}
=
\frac{dG}{ds}.
$$

代入：

$$
-u(x_s,s,t)
+
(t-s)\frac{d}{ds}u(x_s,s,t)
=
-v(x_s,s).
$$

两边乘以：

$$
-1,
$$

得到：

$$
u(x_s,s,t)
-
(t-s)\frac{d}{ds}u(x_s,s,t)
=
v(x_s,s).
$$

移项：

$$
\boxed{
u(x_s,s,t)
=
v(x_s,s)
+
(t-s)\frac{d}{ds}u(x_s,s,t).
}
\tag{11.4}
$$

这就是在当前时间方向下的：

$$
\boxed{\text{MeanFlow Identity}.}
$$

它来自平均速度的严格定义，而不是某种近似。

---

## 12. MeanFlow Identity 中的全导数

公式：

$$
u=v+(t-s)\frac{du}{ds}
$$

中最重要的部分是：

$$
\frac{du}{ds}.
$$

因为：

$$
u=u(x_s,s,t),
$$

所以需要使用多变量链式法则。

### 12.1 全导数展开

$$
\begin{aligned}
\frac{d}{ds}u(x_s,s,t)
={}&
\frac{\partial u}{\partial x}
\frac{dx_s}{ds}\\
&+
\frac{\partial u}{\partial s}
\frac{ds}{ds}\\
&+
\frac{\partial u}{\partial t}
\frac{dt}{ds}.
\end{aligned}
\tag{12.1}
$$

根据 ODE：

$$
\frac{dx_s}{ds}=v(x_s,s).
$$

同时：

$$
\frac{ds}{ds}=1,
$$

因为目标时间固定：

$$
\frac{dt}{ds}=0.
$$

所以：

$$
\boxed{
\frac{d}{ds}u(x_s,s,t)
=
J_u(x_s,s,t)v(x_s,s)
+
\partial_su(x_s,s,t).
}
\tag{12.2}
$$

其中：

$$
J_u=\frac{\partial u}{\partial x}
\in\mathbb R^{d\times d}.
$$

代入 MeanFlow Identity：

$$
\boxed{
u(x_s,s,t)
=
v(x_s,s)
+
(t-s)
\left[
J_uv(x_s,s)+\partial_su
\right].
}
\tag{12.3}
$$

这就是后续训练目标的直接来源。

### 12.2 为什么没有 $\partial_tu$？

因为在推导时：

$$
t
$$

保持不变。

所以：

$$
\frac{dt}{ds}=0.
$$

但起始时间：

$$
s
$$

正在变化。

因此，方向导数中包含：

$$
\partial_su,
$$

而不是：

$$
\partial_tu.
$$

这个细节对于后面 JVP 的实现非常重要。

---

## 13. 使用 JVP 计算全导数

假设神经网络为：

$$
u_\theta(x,s,t).
$$

输出：

$$
u_\theta\in\mathbb R^d.
$$

如果显式构造空间 Jacobian：

$$
J_{u_\theta}\in\mathbb R^{d\times d},
$$

在高维图像生成中计算成本会很高。

但我们只需要：

$$
J_{u_\theta}v+\partial_su_\theta.
$$

因此可以使用 Jacobian-Vector Product，简称 JVP。

### 13.1 将所有输入合并

定义：

$$
y=
\begin{bmatrix}
x\\s\\t
\end{bmatrix}.
$$

因此：

$$
y\in\mathbb R^{d+2}.
$$

神经网络：

$$
f_\theta(y)=u_\theta(x,s,t).
$$

它的 Jacobian：

$$
J_{f_\theta}
=
\begin{bmatrix}
J_{u_\theta}&
\partial_su_\theta&
\partial_tu_\theta
\end{bmatrix}.
$$

现在构造方向向量：

$$
\boxed{
a=
\begin{bmatrix}
v\\1\\0
\end{bmatrix}.
}
\tag{13.1}
$$

那么：

$$
\begin{aligned}
J_{f_\theta}a
={}&
J_{u_\theta}v\\
&+
\partial_su_\theta\cdot1\\
&+
\partial_tu_\theta\cdot0.
\end{aligned}
$$

因此：

$$
\boxed{
J_{f_\theta}a
=
J_{u_\theta}v+\partial_su_\theta.
}
\tag{13.2}
$$

这正是需要的全导数。

所以：

$$
\boxed{
\frac{d}{ds}u_\theta(x_s,s,t)
=
\operatorname{JVP}
\left(
u_\theta,
(x_s,s,t),
(v,1,0)
\right).
}
\tag{13.3}
$$

### 13.2 从方向导数理解

JVP 定义为：

$$
\operatorname{JVP}(f,y,a)
=
\left.
\frac{d}{dh}f(y+ha)
\right|_{h=0}.
$$

对于 MeanFlow：

$$
y+ha
=
\begin{bmatrix}
x+hv\\s+h\\t
\end{bmatrix}.
$$

因此：

$$
\boxed{
\begin{aligned}
&\operatorname{JVP}
\left(
u_\theta,
(x,s,t),
(v,1,0)
\right)\\
&=
\left.
\frac{d}{dh}
u_\theta(x+hv,s+h,t)
\right|_{h=0}.
\end{aligned}
}
\tag{13.4}
$$

其含义是：

- 让状态沿当前速度方向移动；
- 同时让起始时间向前推进；
- 保持目标时间不变；
- 计算平均速度随这一变化的导数。

这与 MeanFlow Identity 的推导完全对应。

---

## 14. MeanFlow 的训练目标

### 14.1 理想目标

如果真实边缘速度：

$$
v(x_s,s)
$$

已知，那么根据 MeanFlow Identity：

$$
u=v+(t-s)(J_uv+\partial_su).
$$

使用网络：

$$
u_\theta
$$

近似真实平均速度。

构造目标：

$$
\boxed{
u_{\mathrm{target}}
=
v(x_s,s)
+
(t-s)
\left[
J_{u_\theta}v(x_s,s)
+
\partial_su_\theta
\right].
}
\tag{14.1}
$$

训练：

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
\left\|
u_\theta(x_s,s,t)
-
\operatorname{sg}(u_{\mathrm{target}})
\right\|^2
\right].
}
\tag{14.2}
$$

这里的目标由当前网络自身的导数构造。

因此，训练过程不需要显式计算真实平均速度：

$$
u(x_s,s,t).
$$

### 14.2 使用条件速度替换边缘速度

但真实边缘速度：

$$
v(x_s,s)
=
\mathbb E[x_1-x_0\mid x_s]
$$

无法直接计算。

训练时已知：

$$
x_0,x_1.
$$

因此可以计算条件速度：

$$
\boxed{
V=x_1-x_0.
}
\tag{14.3}
$$

用它替换边缘速度，得到：

$$
\boxed{
u_{\mathrm{target}}
=
V+
(t-s)
\left[
J_{u_\theta}V+\partial_su_\theta
\right].
}
\tag{14.4}
$$

使用 JVP：

$$
\boxed{
u_{\mathrm{target}}
=
V+
(t-s)
\operatorname{JVP}
\left(
u_\theta,
(x_s,s,t),
(V,1,0)
\right).
}
\tag{14.5}
$$

最终损失：

$$
\boxed{
\mathcal L_{\mathrm{MF}}
=
\mathbb E_{x_0,x_1,s,t}
\left[
\left\|
u_\theta(x_s,s,t)
-
\operatorname{sg}(u_{\mathrm{target}})
\right\|^2
\right].
}
\tag{14.6}
$$

这就是新时间约定下的完整训练公式。

---

## 15. 为什么条件速度能够替换边缘速度？

这一点可以通过条件期望严格推导。

定义：

$$
V=x_1-x_0.
$$

边缘速度：

$$
v(x,s)
=
\mathbb E[V\mid x_s=x].
$$

设：

$$
\Delta=t-s.
$$

随机训练目标：

$$
\hat u_{\mathrm{target}}
=
V+\Delta(J_{u_\theta}V+\partial_su_\theta).
$$

整理：

$$
\boxed{
\hat u_{\mathrm{target}}
=
(I+\Delta J_{u_\theta})V
+
\Delta\partial_su_\theta.
}
\tag{15.1}
$$

现在固定：

$$
x_s=x,\quad s,\quad t,\quad\theta.
$$

此时：

$$
J_{u_\theta}
$$

和：

$$
\partial_su_\theta
$$

都是确定的。

因此：

$$
\begin{aligned}
&\mathbb E[
\hat u_{\mathrm{target}}
\mid x_s=x,s,t
]\\
={}&
(I+\Delta J_{u_\theta})
\mathbb E[V\mid x_s=x]\\
&+
\Delta\partial_su_\theta.
\end{aligned}
$$

代入：

$$
\mathbb E[V\mid x_s=x]=v(x,s),
$$

得到：

$$
\boxed{
\begin{aligned}
\mathbb E[
\hat u_{\mathrm{target}}
\mid x_s=x,s,t
]
={}&
v(x,s)\\
&+
\Delta
\left[
J_{u_\theta}v(x,s)
+
\partial_su_\theta
\right].
\end{aligned}
}
\tag{15.2}
$$

这正是使用真实边缘速度构造的 MeanFlow 目标。

因此：

$$
\boxed{
\text{条件速度目标的条件期望}
=
\text{边缘速度目标}.
}
\tag{15.3}
$$

这说明，条件速度：

$$
V=x_1-x_0
$$

可以用于 MeanFlow 训练。

---

## 16. Stop-gradient 的作用

MeanFlow 的训练目标：

$$
u_{\mathrm{target}}
=
V+\Delta D_su_\theta
$$

依赖网络参数：

$$
\theta.
$$

如果直接最小化：

$$
\|u_\theta-u_{\mathrm{target}}\|^2
$$

并对所有项完整反向传播，那么梯度会穿过：

$$
D_su_\theta.
$$

由于：

$$
D_su_\theta
=
J_{u_\theta}V+\partial_su_\theta,
$$

它已经包含网络输出关于输入的一阶导数。

再次对网络参数求导，就会涉及混合二阶导数。

论文采用：

$$
\operatorname{sg}(u_{\mathrm{target}}).
$$

即：

$$
\frac{\partial\operatorname{sg}(u_{\mathrm{target}})}
{\partial\theta}=0.
$$

于是：

$$
\boxed{
\nabla_\theta\mathcal L
=
2J_\theta u_\theta^\top
\left[
u_\theta-\operatorname{sg}(u_{\mathrm{target}})
\right].
}
\tag{16.1}
$$

这样，目标分支不再参与反向传播。

但在线网络输出：

$$
u_\theta
$$

仍然正常更新。

---

## 17. 为什么训练时需要 $s=t$？

MeanFlow 的训练不仅包含：

$$
s<t,
$$

还会让部分样本满足：

$$
s=t.
$$

### 17.1 当 $s=t$

此时：

$$
\Delta=t-s=0.
$$

训练目标：

$$
u_{\mathrm{target}}
=
V+\Delta D_su_\theta
$$

退化为：

$$
\boxed{
u_{\mathrm{target}}=V.
}
\tag{17.1}
$$

因此：

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
\|u_\theta(x_s,s,s)-V\|^2
\right].
}
\tag{17.2}
$$

这就是标准 Flow Matching。

其最优预测：

$$
\boxed{
u_\theta(x,s,s)
=
v(x,s).
}
\tag{17.3}
$$

所以：

$$
s=t
$$

的样本用于学习瞬时速度。

### 17.2 当 $s<t$

此时：

$$
\Delta>0.
$$

目标：

$$
u_{\mathrm{target}}
=
V+\Delta D_su_\theta.
$$

网络需要学习有限区间平均速度。

因此：

$$
\boxed{
\begin{aligned}
s=t &: \text{学习瞬时速度};\\
s<t &: \text{学习平均速度}.
\end{aligned}
}
\tag{17.4}
$$

论文 ImageNet 实验中，约 25% 的训练样本使用：

$$
s<t,
$$

其余使用：

$$
s=t.
$$

实验表明，两种监督方式混合训练效果最好。

---

## 18. MeanFlow 的一步与多步生成

### 18.1 一步生成

根据：

$$
x_t=x_s+(t-s)u(x_s,s,t),
$$

令：

$$
s=0,\qquad t=1,
$$

得到：

$$
\boxed{
x_1=x_0+u(x_0,0,1).
}
\tag{18.1}
$$

用神经网络近似：

$$
\boxed{
\hat x_1=x_0+u_\theta(x_0,0,1).
}
\tag{18.2}
$$

整个生成过程只需要一次网络推理：

$$
\boxed{
\mathrm{NFE}=1.
}
\tag{18.3}
$$

### 18.2 两步生成

选择：

$$
m\in(0,1).
$$

第一步：

$$
\boxed{
x_m=x_0+m\,u_\theta(x_0,0,m).
}
\tag{18.4}
$$

第二步：

$$
\boxed{
x_1=x_m+(1-m)u_\theta(x_m,m,1).
}
\tag{18.5}
$$

因此：

$$
\mathrm{NFE}=2.
$$

### 18.3 多步生成

设：

$$
0=t_0<t_1<\cdots<t_N=1.
$$

每一步：

$$
\boxed{
x_{t_{i+1}}
=
x_{t_i}
+
(t_{i+1}-t_i)
u_\theta(x_{t_i},t_i,t_{i+1}).
}
\tag{18.6}
$$

需要注意：

这与普通 Euler 积分形式相似，但并不相同。

Euler 使用：

$$
v_\theta(x_{t_i},t_i).
$$

MeanFlow 使用：

$$
u_\theta(x_{t_i},t_i,t_{i+1}).
$$

前者是瞬时速度。

后者是区间平均速度。

如果平均速度预测完全准确，那么无论时间区间多大，都可以准确得到对应 ODE 的终点。

---

# 第四部分：MeanFlow with Guidance

对应论文第 4.2 节。

## 19. Classifier-Free Guidance 的定义

设条件为：

$$
c.
$$

条件边缘速度：

$$
v(x,s\mid c).
$$

无条件边缘速度：

$$
v(x,s).
$$

CFG 定义：

$$
\boxed{
v^{\mathrm{cfg}}(x,s\mid c)
=
\omega v(x,s\mid c)
+
(1-\omega)v(x,s).
}
\tag{19.1}
$$

其中：

$$
\omega
$$

为 CFG 强度。

例如：

$$
\omega=3,
$$

则：

$$
v^{\mathrm{cfg}}
=
3v_{\mathrm{cond}}-2v_{\mathrm{uncond}}.
$$

普通 Flow Matching 推理时，需要分别预测条件速度和无条件速度。

因此，每个积分步通常需要两次网络推理。

MeanFlow 希望在训练阶段直接学习 CFG 速度场对应的平均速度。

---

## 20. CFG 平均速度的定义

定义 CFG ODE：

$$
\boxed{
\frac{dx_s}{ds}
=
v^{\mathrm{cfg}}(x_s,s\mid c).
}
\tag{20.1}
$$

对于：

$$
s<t,
$$

定义平均速度：

$$
\boxed{
u^{\mathrm{cfg}}(x_s,s,t\mid c)
=
\frac1{t-s}
\int_s^t
v^{\mathrm{cfg}}(x_\tau,\tau\mid c)
\,d\tau.
}
\tag{20.2}
$$

这里：

$$
x_\tau
$$

是 CFG ODE 自身的轨迹。

因此，根据 MeanFlow Identity：

$$
\boxed{
u^{\mathrm{cfg}}
=
v^{\mathrm{cfg}}
+
(t-s)
\frac{d}{ds}u^{\mathrm{cfg}}.
}
\tag{20.3}
$$

展开全导数：

$$
\boxed{
u^{\mathrm{cfg}}
=
v^{\mathrm{cfg}}
+
(t-s)
\left[
J_{u^{\mathrm{cfg}}}v^{\mathrm{cfg}}
+
\partial_su^{\mathrm{cfg}}
\right].
}
\tag{20.4}
$$

这意味着，MeanFlow 的训练方法可以直接应用于 CFG 速度场。

关键问题变成：

**如何在训练时构造 CFG 的瞬时速度监督？**

---

## 21. CFG 训练目标的构造

理论上：

$$
v^{\mathrm{cfg}}
=
\omega v_{\mathrm{cond}}
+
(1-\omega)v_{\mathrm{uncond}}.
$$

条件速度可以通过样本：

$$
V=x_1-x_0
$$

进行估计。

因为：

$$
\mathbb E[V\mid x_s=x,c]
=
v_{\mathrm{cond}}(x,s,c).
$$

无条件速度则可以利用 MeanFlow 网络在对角线上的输出：

$$
u_\theta(x_s,s,s).
$$

根据平均速度的边界性质：

$$
u(x,s,s)=v(x,s).
$$

因此，可以构造：

$$
\boxed{
\tilde V
=
\omega V
+
(1-\omega)u_\theta(x_s,s,s).
}
\tag{21.1}
$$

这里无条件分支不输入类别条件。

在理想情况下：

$$
u_\theta(x_s,s,s)\approx v_{\mathrm{uncond}}(x_s,s).
$$

因此：

$$
\tilde V
$$

近似 CFG 瞬时速度的随机监督。

### 21.1 代入 MeanFlow Identity

定义：

$$
\Delta=t-s.
$$

那么：

$$
\boxed{
u_{\mathrm{target}}^{\mathrm{cfg}}
=
\tilde V
+
\Delta
\left[
J_{u_\theta^{\mathrm{cfg}}}\tilde V
+
\partial_su_\theta^{\mathrm{cfg}}
\right].
}
\tag{21.2}
$$

使用 JVP：

$$
\boxed{
\begin{aligned}
u_{\mathrm{target}}^{\mathrm{cfg}}
={}&
\tilde V\\
&+
(t-s)
\operatorname{JVP}
\left(
u_\theta^{\mathrm{cfg}},
(x_s,s,t),
(\tilde V,1,0)
\right).
\end{aligned}
}
\tag{21.3}
$$

训练损失：

$$
\boxed{
\mathcal L_{\mathrm{CFG}}
=
\mathbb E
\left[
\left\|
u_\theta^{\mathrm{cfg}}(x_s,s,t\mid c)
-
\operatorname{sg}
(u_{\mathrm{target}}^{\mathrm{cfg}})
\right\|^2
\right].
}
\tag{21.4}
$$

实际构造目标时，需要将用于估计无条件速度的预测作为停止梯度的目标组成部分，避免额外的目标分支梯度。

---

## 22. 为什么 CFG 仍然可以一步生成？

训练完成后，网络直接预测：

$$
u_\theta^{\mathrm{cfg}}(x_s,s,t\mid c).
$$

推理时：

$$
s=0,\qquad t=1.
$$

所以：

$$
\boxed{
\hat x_1
=
x_0+
u_\theta^{\mathrm{cfg}}(x_0,0,1\mid c).
}
\tag{22.1}
$$

只需要一次网络调用。

因此：

$$
\boxed{
\mathrm{NFE}=1.
}
\tag{22.2}
$$

核心原因是：

**CFG 已经被吸收到网络学习的平均速度场中，而不是在推理时通过两次网络输出进行组合。**

---

## 23. Improved CFG：改进的训练目标

论文附录 B.1 进一步引入参数：

$$
\kappa.
$$

目的是同时使用条件和无条件的对角线速度预测。

在我们的时间方向下，构造：

$$
\boxed{
\begin{aligned}
\tilde V
={}&
\omega V\\
&+
\kappa u_\theta^{\mathrm{cfg}}(x_s,s,s\mid c)\\
&+
(1-\omega-\kappa)
u_\theta^{\mathrm{cfg}}(x_s,s,s).
\end{aligned}
}
\tag{23.1}
$$

其中：

- 第一项：样本级条件速度；
- 第二项：条件 CFG 对角线速度；
- 第三项：无条件对角线速度。

### 23.1 为什么引入 $\kappa$？

考虑理想情况下：

$$
u^{\mathrm{cfg}}(x,s,s\mid c)
=
v^{\mathrm{cfg}}(x,s\mid c),
$$

以及：

$$
u^{\mathrm{cfg}}(x,s,s)
=
v(x,s).
$$

那么：

$$
v^{\mathrm{cfg}}
=
\omega v_{\mathrm{cond}}
+
\kappa v^{\mathrm{cfg}}
+
(1-\omega-\kappa)v_{\mathrm{uncond}}.
$$

将含有：

$$
v^{\mathrm{cfg}}
$$

的项移到左边：

$$
(1-\kappa)v^{\mathrm{cfg}}
=
\omega v_{\mathrm{cond}}
+
(1-\omega-\kappa)v_{\mathrm{uncond}}.
$$

两边除以：

$$
1-\kappa,
$$

得到：

$$
v^{\mathrm{cfg}}
=
\frac{\omega}{1-\kappa}v_{\mathrm{cond}}
+
\frac{1-\omega-\kappa}{1-\kappa}
v_{\mathrm{uncond}}.
$$

注意：

$$
\frac{1-\omega-\kappa}{1-\kappa}
=
1-\frac{\omega}{1-\kappa}.
$$

因此：

$$
\boxed{
v^{\mathrm{cfg}}
=
\omega'v_{\mathrm{cond}}
+
(1-\omega')v_{\mathrm{uncond}},
}
\tag{23.2}
$$

其中：

$$
\boxed{
\omega'=\frac{\omega}{1-\kappa}.
}
\tag{23.3}
$$

也就是说，这个改进形式仍然对应标准 CFG，只是有效 guidance scale 为：

$$
\omega'.
$$

论文在实验中发现，适当引入：

$$
\kappa
$$

能够进一步改善生成质量。

---

# 第五部分：Design Decisions

对应论文第 4.3 节。

## 24. 自适应损失权重

MeanFlow 的基础训练损失为：

$$
\mathcal L
=
\|u_\theta-u_{\mathrm{target}}\|^2.
$$

定义残差：

$$
\boxed{
e=u_\theta-\operatorname{sg}(u_{\mathrm{target}}).
}
\tag{24.1}
$$

普通平方误差：

$$
\mathcal L=\|e\|^2.
$$

论文研究一种幂次损失：

$$
\boxed{
\mathcal L_\gamma
=
\|e\|^{2\gamma}.
}
\tag{24.2}
$$

### 24.1 推导自适应权重

令：

$$
q=\|e\|^2.
$$

那么：

$$
\mathcal L_\gamma=q^\gamma.
$$

对模型参数求导：

$$
\nabla_\theta\mathcal L_\gamma
=
\gamma q^{\gamma-1}\nabla_\theta q.
$$

定义：

$$
p=1-\gamma.
$$

则：

$$
\gamma-1=-p.
$$

因此：

$$
\nabla_\theta\mathcal L_\gamma
=
\gamma
\frac1{q^p}
\nabla_\theta q.
$$

忽略可吸收到整体学习率中的常数：

$$
\gamma,
$$

得到权重：

$$
w\propto\frac1{q^p}.
$$

为了避免：

$$
q\rightarrow0
$$

时权重发散，加入：

$$
c>0.
$$

最终：

$$
\boxed{
w=
\frac1{(\|e\|^2+c)^p}.
}
\tag{24.3}
$$

实际训练：

$$
\boxed{
\mathcal L_{\mathrm{weighted}}
=
\operatorname{sg}(w)\|e\|^2.
}
\tag{24.4}
$$

### 24.2 参数 $p$ 的作用

当：

$$
p=0,
$$

有：

$$
w=1.
$$

退化为普通 MSE。

当：

$$
p>0,
$$

残差越大，权重越小。

例如：

$$
p=1,
$$

则：

$$
w=\frac1{\|e\|^2+c}.
$$

这会抑制大残差样本对梯度的影响。

论文的 ImageNet 消融实验中：

$$
p=1
$$

效果最好。

---

## 25. 时间采样策略

MeanFlow 网络：

$$
u_\theta(x_s,s,t)
$$

需要两个时间变量：

$$
s,t.
$$

要求：

$$
0\leq s\leq t\leq1.
$$

### 25.1 均匀采样

采样：

$$
a,b\overset{\mathrm{iid}}{\sim}\mathcal U(0,1).
$$

然后：

$$
\boxed{
s=\min(a,b),
\qquad
t=\max(a,b).
}
\tag{25.1}
$$

这样可以保证：

$$
s\leq t.
$$

### 25.2 Logit-Normal 采样

先采样：

$$
y\sim\mathcal N(\mu,\sigma^2).
$$

通过 sigmoid：

$$
\boxed{
q=\frac1{1+e^{-y}}.
}
\tag{25.2}
$$

得到：

$$
q\in(0,1).
$$

分别独立采样：

$$
q_1,q_2,
$$

然后：

$$
\boxed{
s=\min(q_1,q_2),
\qquad
t=\max(q_1,q_2).
}
\tag{25.3}
$$

此外，训练时还会人为设置一定比例的样本：

$$
s=t.
$$

这些样本用于瞬时速度监督。

---

## 26. 时间条件编码

最直接的网络输入为：

$$
u_\theta(x,s,t).
$$

但由于：

$$
\Delta=t-s,
$$

所以：

$$
(s,t)
$$

也可以表示为：

$$
(s,\Delta).
$$

定义：

$$
\boxed{
\tilde u_\theta(x,s,\Delta)
=
u_\theta(x,s,s+\Delta).
}
\tag{26.1}
$$

其中：

$$
\Delta=t-s.
$$

### 26.1 为什么使用 $(s,\Delta)$？

因为：

$$
s
$$

表示当前时间。

而：

$$
\Delta
$$

直接表示需要预测的时间跨度。

当：

$$
\Delta=0,
$$

网络预测瞬时速度。

当：

$$
\Delta>0,
$$

网络预测平均速度。

这种表示能够直接体现平均速度与时间跨度的关系。

### 26.2 变换时间参数后，JVP 方向如何变化？

这是一个非常重要的实现细节。

原始输入：

$$
(x,s,t).
$$

固定：

$$
t
$$

时，JVP 方向为：

$$
\boxed{
(V,1,0).
}
\tag{26.2}
$$

现在改成：

$$
(x,s,\Delta),
$$

其中：

$$
\Delta=t-s.
$$

因为目标时间固定：

$$
\frac{dt}{ds}=0,
$$

所以：

$$
\frac{d\Delta}{ds}
=
\frac{d(t-s)}{ds}
=
-1.
$$

因此：

$$
\begin{aligned}
\frac{d}{ds}\tilde u_\theta(x_s,s,\Delta)
={}&
J_x\tilde u_\theta V\\
&+
\partial_s\tilde u_\theta\\
&-
\partial_\Delta\tilde u_\theta.
\end{aligned}
$$

所以新的 JVP 方向应为：

$$
\boxed{
(V,1,-1).
}
\tag{26.3}
$$

注意：

**JVP 的方向必须与网络实际使用的时间坐标保持一致。**

在原始输入：

$$
(x,s,t)
$$

下使用：

$$
(V,1,0).
$$

在新输入：

$$
(x,s,t-s)
$$

下使用：

$$
(V,1,-1).
$$

两者在数学上完全等价。

---

# 第六部分：Experiments

对应论文第 5 节。

本节的实验数值沿用论文原始报告。由于我们只改变时间坐标约定，没有改变生成任务、模型或评价指标，因此实验结果不需要变化。

## 27. 实验设置

论文主要在两个数据集上验证 MeanFlow。

### 27.1 ImageNet 256×256

主要验证：

- 一步生成质量；
- CFG 的作用；
- 模型规模扩展；
- 时间采样策略；
- JVP 方向；
- 自适应损失权重。

模型采用 Transformer 主干，并在预训练 VAE 的潜在空间中进行生成。

模型规模包括：

$$
\text{B/2、M/2、L/2、XL/2}.
$$

主要模型训练 240 epochs。

消融实验采用较小模型 B/4，训练 80 epochs。

### 27.2 CIFAR-10

主要验证无条件一步生成。

采用约 55M 参数的 U-Net。

主要评价指标：

$$
\mathrm{FID}
$$

以及：

$$
\mathrm{NFE}.
$$

其中：

$$
\mathrm{NFE}
$$

表示推理时网络函数调用次数。

---

## 28. 消融实验

### 28.1 平均速度训练比例

论文研究：

$$
s<t
$$

的训练样本比例。

| $s<t$ 比例 | 1-NFE FID ↓ |
|---|---:|
| 0% | 328.91 |
| **25%** | **61.06** |
| 50% | 63.14 |
| 100% | 67.32 |

当：

$$
s=t
$$

始终成立时：

$$
u_{\mathrm{target}}=V.
$$

此时网络只学习瞬时速度。

直接进行一步生成，效果很差。

当引入：

$$
s<t
$$

的样本后，网络开始学习有限区间平均速度，一步生成质量明显提升。

实验中：

$$
25\%
$$

取得最好的结果。

说明：

**瞬时速度监督和平均速度监督具有互补作用。**

### 28.2 JVP 方向

原论文使用的是反向时间坐标，因此原始消融方向不能直接照搬到我们现在的符号中。

在当前约定下，正确方向为：

$$
\boxed{
(V,1,0).
}
$$

原论文的 JVP 消融结果如下：

| 原论文 JVP 方向 | FID ↓ |
|---|---:|
| **正确方向** | **61.06** |
| 仅空间方向 | 268.06 |
| 错误的时间方向一 | 329.22 |
| 错误的时间方向二 | 137.96 |

这里保留原论文数值，但用含义描述错误方向，避免将反向时间坐标下的向量直接误认为当前坐标下的向量。

在我们的约定下：

$$
(x,s,t)
$$

的正确 JVP 方向为：

$$
(V,1,0).
$$

其余方向必须通过对应的时间坐标变换才能正确解释。

实验表明，正确计算 MeanFlow Identity 中的方向导数非常重要。

### 28.3 时间条件编码

原论文研究了不同时间表示方式。

对应到我们的时间约定：

| 时间条件 | FID ↓ |
|---|---:|
| $(s,t)$ | 61.75 |
| **$(s,t-s)$** | **61.06** |
| $(s,t,t-s)$ | 63.98 |
| $t-s$ | 63.13 |

实验中：

$$
(s,t-s)
$$

效果最好。

这里的时间参数已经根据生成方向进行了对应转换。

### 28.4 时间采样

| 时间采样 | FID ↓ |
|---|---:|
| Uniform | 65.90 |
| Logit-normal $(-0.2,1.0)$ | 63.83 |
| Logit-normal $(-0.2,1.2)$ | 64.72 |
| **Logit-normal $(-0.4,1.0)$** | **61.06** |
| Logit-normal $(-0.4,1.2)$ | 61.79 |

这里列出的是论文原始采样配置及结果。

**注意：** 如果将原论文的时间变量严格反向映射为当前约定：

$$
s_{\mathrm{new}}=1-t_{\mathrm{old}},
$$

那么 logit-normal 的位置参数也需要变号：

$$
\mu_{\mathrm{new}}=-\mu_{\mathrm{old}}.
$$

这是因为：

$$
1-\sigma(y)=\sigma(-y).
$$

因此，表中的 $\mu$ 是原论文报告的参数；若在当前时间坐标下复现实验，应使用对应的变换后参数。

### 28.5 自适应损失权重

| $p$ | FID ↓ |
|---|---:|
| 0 | 79.75 |
| 0.5 | 63.98 |
| **1.0** | **61.06** |
| 1.5 | 66.57 |
| 2.0 | 69.19 |

其中：

$$
w=\frac1{(\|e\|^2+c)^p}.
$$

实验中：

$$
p=1
$$

效果最好。

### 28.6 CFG 强度

| $\omega$ | FID ↓ |
|---|---:|
| 1.0 | 61.06 |
| 1.5 | 33.33 |
| 2.0 | 20.15 |
| **3.0** | **15.53** |
| 5.0 | 20.75 |

适当提高 CFG 强度可以显著改善生成质量。

但 CFG 强度并非越大越好。

---

## 29. ImageNet 主要实验结果

| 模型 | 参数量 | NFE | FID ↓ |
|---|---:|---:|---:|
| MeanFlow-B/2 | 131M | 1 | 6.17 |
| MeanFlow-M/2 | 308M | 1 | 5.01 |
| MeanFlow-L/2 | 459M | 1 | 3.84 |
| **MeanFlow-XL/2** | **676M** | **1** | **3.43** |
| MeanFlow-XL/2 | 676M | 2 | 2.93 |
| MeanFlow-XL/2+ | 676M | 2 | 2.20 |

可以看到：

随着模型规模增加，一步生成质量持续改善。

此外，MeanFlow 也支持多步采样。

两步采样可以进一步改善生成质量。

---

## 30. CIFAR-10 实验

论文在 CIFAR-10 上进行无条件生成。

采用约 55M 参数的 U-Net。

主要训练配置：

| 项目 | 配置 |
|---|---|
| Optimizer | Adam |
| Learning rate | $6\times10^{-4}$ |
| Batch size | 1024 |
| Training iterations | 800K |
| EMA decay | 0.99995 |
| $s<t$ 比例 | 75% |
| Loss weight $p$ | 0.75 |

最终：

$$
\boxed{
\mathrm{FID}=2.92,
\qquad
\mathrm{NFE}=1.
}
\tag{30.1}
$$

说明 MeanFlow 不仅适用于 Transformer 潜在空间生成，也能够应用于像素空间 U-Net。

---

# 第七部分：论文结论

MeanFlow 的核心贡献，是将生成模型的预测目标从瞬时速度：

$$
v(x_s,s)
$$

扩展为有限时间平均速度：

$$
u(x_s,s,t).
$$

平均速度定义：

$$
\boxed{
u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
}
$$

通过对起始时间求导：

$$
\boxed{
u
=
v+(t-s)(J_uv+\partial_su).
}
$$

利用 JVP 构造训练目标：

$$
\boxed{
u_{\mathrm{target}}
=
V+
(t-s)
\operatorname{JVP}
\left(
u_\theta,
(x_s,s,t),
(V,1,0)
\right).
}
$$

最终：

$$
\boxed{
x_1=x_0+u_\theta(x_0,0,1).
}
$$

从而实现：

$$
\boxed{
\mathrm{NFE}=1.
}
$$

**MeanFlow 最重要的思想，是让网络直接学习有限时间区间内的净位移，而不是依赖多次瞬时速度预测来累计位移。**

---

# 附录：MeanFlow 使用的高级数学基础

本附录主要补充论文涉及的数学知识，包括：

1. 非自治 ODE 与两参数流映射
2. 连续性方程与概率分布推送
3. 物质导数与输运型 PDE
4. MeanFlow Identity 的充分性和唯一性
5. 平均速度的高阶渐近展开
6. JVP 与自动微分
7. Stop-gradient 与半梯度优化
8. 条件速度目标的方差结构
9. 时间反演与公式变换

---

## A.1 非自治 ODE 与两参数流映射

### A.1.1 非自治 ODE

考虑：

$$
\boxed{
\frac{dx_t}{dt}=v(x_t,t).
}
\tag{A.1}
$$

由于速度场显式依赖时间：

$$
t,
$$

因此称为非自治 ODE。

给定：

$$
x_s=x,
$$

在适当的正则性条件下，可以唯一确定之后的状态：

$$
x_t.
$$

定义流映射：

$$
\boxed{
\Phi_{t\leftarrow s}(x)=x_t.
}
\tag{A.2}
$$

### A.1.2 流映射的组合性质

考虑：

$$
s<m<t.
$$

直接从 $s$ 积分到 $t$：

$$
x_t=\Phi_{t\leftarrow s}(x_s).
$$

也可以先积分到 $m$：

$$
x_m=\Phi_{m\leftarrow s}(x_s),
$$

再积分到 $t$：

$$
x_t=\Phi_{t\leftarrow m}(x_m).
$$

因此：

$$
\boxed{
\Phi_{t\leftarrow s}
=
\Phi_{t\leftarrow m}
\circ
\Phi_{m\leftarrow s}.
}
\tag{A.3}
$$

同时：

$$
\boxed{
\Phi_{s\leftarrow s}=\mathrm{Id}.
}
\tag{A.4}
$$

这里：

$$
\mathrm{Id}(x)=x.
$$

### A.1.3 MeanFlow 与流映射

MeanFlow 定义：

$$
u(x,s,t)
=
\frac{
\Phi_{t\leftarrow s}(x)-x
}{
t-s
}.
$$

所以：

$$
\boxed{
\Phi_{t\leftarrow s}(x)
=
x+(t-s)u(x,s,t).
}
\tag{A.5}
$$

这说明，平均速度与有限时间流映射之间可以相互转换。

在数学上，MeanFlow 学习的是对流映射位移的一种参数化。

---

## A.2 连续性方程与概率分布推送

### A.2.1 推送测度

设：

$$
X_0\sim p_0.
$$

通过确定性映射：

$$
T:\mathbb R^d\rightarrow\mathbb R^d,
$$

得到：

$$
X_1=T(X_0).
$$

那么 $X_1$ 的分布记为：

$$
\boxed{
p_1=T_\#p_0.
}
\tag{A.6}
$$

其中：

$$
T_\#
$$

表示概率分布的推送操作。

对于 ODE：

$$
T=\Phi_{1\leftarrow0}.
$$

因此：

$$
\boxed{
p_1=
(\Phi_{1\leftarrow0})_\#p_0.
}
\tag{A.7}
$$

### A.2.2 连续性方程

设：

$$
p_t(x)
$$

是中间状态的概率密度。

沿 ODE：

$$
\frac{dx_t}{dt}=v(x_t,t),
$$

概率密度满足：

$$
\boxed{
\partial_tp_t(x)
+
\nabla\cdot[p_t(x)v(x,t)]
=
0.
}
\tag{A.8}
$$

这是概率质量守恒方程。

它说明，概率质量随着速度场运动，但不会凭空产生或消失。

### A.2.3 推导

取任意光滑测试函数：

$$
\varphi(x).
$$

有：

$$
\mathbb E[\varphi(X_t)]
=
\int\varphi(x)p_t(x)\,dx.
$$

对时间求导：

$$
\frac{d}{dt}
\mathbb E[\varphi(X_t)]
=
\mathbb E[
\nabla\varphi(X_t)^\top v(X_t,t)
].
$$

写成积分：

$$
=
\int
\nabla\varphi(x)^\top
v(x,t)p_t(x)\,dx.
$$

使用分部积分：

$$
=
-\int
\varphi(x)
\nabla\cdot[p_t(x)v(x,t)]
\,dx.
$$

另一方面：

$$
\frac{d}{dt}
\mathbb E[\varphi(X_t)]
=
\int
\varphi(x)\partial_tp_t(x)\,dx.
$$

因此：

$$
\int
\varphi(x)
\left[
\partial_tp_t+
\nabla\cdot(p_tv)
\right]dx=0.
$$

由于对任意测试函数都成立：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tv)=0.
}
\tag{A.9}
$$

### A.2.4 MeanFlow 与概率推送

如果平均速度精确，那么：

$$
x_1=x_0+u(x_0,0,1)
$$

与原始 ODE 流映射完全相同。

因此：

$$
\boxed{
p_1=
\left[
\mathrm{Id}+u(\cdot,0,1)
\right]_\#p_0.
}
\tag{A.10}
$$

这说明，MeanFlow 的精确一步生成与原始 ODE 的概率输运等价。

---

## A.3 物质导数与输运算子

### A.3.1 欧拉描述与拉格朗日描述

考虑一个场：

$$
u(x,s,t).
$$

固定空间位置：

$$
x,
$$

观察它随起始时间 $s$ 的变化，对应偏导数：

$$
\partial_su.
$$

但如果沿着运动轨迹：

$$
x_s,
$$

观察平均速度的变化，则需要全导数：

$$
\frac{d}{ds}u(x_s,s,t).
$$

两者不同。

因为运动轨迹中的空间位置也在变化。

### A.3.2 物质导数

定义沿速度场：

$$
v
$$

的物质导数：

$$
\boxed{
D_s^vu
=
\partial_su+(v\cdot\nabla)u.
}
\tag{A.11}
$$

对于向量值函数：

$$
u:\mathbb R^d\rightarrow\mathbb R^d,
$$

有：

$$
(v\cdot\nabla)u=J_uv.
$$

因此：

$$
\boxed{
D_s^vu
=
\partial_su+J_uv.
}
\tag{A.12}
$$

MeanFlow Identity 可以写成：

$$
\boxed{
u=v+(t-s)D_s^vu.
}
\tag{A.13}
$$

整理：

$$
\boxed{
u-(t-s)D_s^vu=v.
}
\tag{A.14}
$$

这是一个沿速度场：

$$
v
$$

定义的一阶输运型 PDE。

### A.3.3 特征线方法

考虑：

$$
u-(t-s)D_s^vu=v.
$$

沿 ODE 特征线：

$$
\frac{dx_s}{ds}=v(x_s,s),
$$

有：

$$
D_s^vu
=
\frac{du(x_s,s,t)}{ds}.
$$

因此 PDE 化为：

$$
u-(t-s)\frac{du}{ds}=v.
$$

两边乘以：

$$
-1,
$$

得到：

$$
-u+(t-s)\frac{du}{ds}=-v.
$$

注意：

$$
\frac{d}{ds}[(t-s)u]
=
-u+(t-s)\frac{du}{ds}.
$$

因此：

$$
\boxed{
\frac{d}{ds}[(t-s)u]=-v.
}
\tag{A.15}
$$

从 $s$ 积分到 $t$：

$$
\int_s^t
\frac{d}{d\tau}[(t-\tau)u]\,d\tau
=
-\int_s^t v(x_\tau,\tau)\,d\tau.
$$

左边：

$$
\left[(t-\tau)u\right]_{\tau=s}^{\tau=t}.
$$

若平均速度在对角线附近有界，则：

$$
\lim_{\tau\to t^-}
(t-\tau)u=0.
$$

因此：

$$
-(t-s)u(x_s,s,t)
=
-\int_s^t v(x_\tau,\tau)\,d\tau.
$$

两边乘以：

$$
-1,
$$

得到：

$$
(t-s)u(x_s,s,t)
=
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

最终：

$$
\boxed{
u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
}
\tag{A.16}
$$

这说明 MeanFlow Identity 与平均速度的积分定义相互对应。

---

## A.4 MeanFlow Identity 的充分性与唯一性

设某个函数：

$$
\tilde u(x,s,t)
$$

满足：

$$
\boxed{
\tilde u
=
v+(t-s)D_s^v\tilde u.
}
\tag{A.17}
$$

我们希望证明：

$$
\tilde u=u.
$$

### A.4.1 沿特征线化为 ODE

沿：

$$
\frac{dx_s}{ds}=v(x_s,s),
$$

有：

$$
\tilde u
=
v+(t-s)\frac{d\tilde u}{ds}.
$$

移项：

$$
-\tilde u+(t-s)\frac{d\tilde u}{ds}
=
-v.
$$

因此：

$$
\frac{d}{ds}[(t-s)\tilde u]
=
-v.
$$

从 $s$ 积分到 $t$：

$$
\begin{aligned}
&\lim_{\tau\to t^-}
(t-\tau)\tilde u(x_\tau,\tau,t)\\
&-
(t-s)\tilde u(x_s,s,t)
=
-\int_s^t v(x_\tau,\tau)\,d\tau.
\end{aligned}
$$

如果：

$$
\tilde u
$$

在：

$$
s\rightarrow t
$$

附近有界，则第一项为零。

因此：

$$
(t-s)\tilde u
=
\int_s^t v(x_\tau,\tau)\,d\tau.
$$

得到：

$$
\boxed{
\tilde u(x_s,s,t)
=
\frac1{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau.
}
\tag{A.18}
$$

这正是真实平均速度。

### A.4.2 为什么有界性重要？

考虑齐次方程：

$$
h=(t-s)D_s^vh.
$$

沿特征线：

$$
h=(t-s)\frac{dh}{ds}.
$$

整理：

$$
-h+(t-s)\frac{dh}{ds}=0.
$$

因此：

$$
\frac{d}{ds}[(t-s)h]=0.
$$

所以：

$$
(t-s)h=C.
$$

得到：

$$
\boxed{
h=\frac{C}{t-s}.
}
\tag{A.19}
$$

当：

$$
s\rightarrow t,
$$

如果：

$$
C\neq0,
$$

那么：

$$
h\rightarrow\infty.
$$

因此，要求解在对角线附近有界，就必须：

$$
C=0.
$$

这排除了额外的奇异解。

所以在适当的正则性和边界条件下，MeanFlow Identity 能够唯一确定平均速度。

---

## A.5 平均速度的高阶渐近展开

这部分可以帮助理解平均速度为什么包含比瞬时速度更丰富的信息。

定义：

$$
h=t-s.
$$

我们希望研究：

$$
h\rightarrow0
$$

时平均速度的行为。

### A.5.1 沿轨迹展开瞬时速度

定义：

$$
g(\tau)=v(x_\tau,\tau).
$$

沿 ODE：

$$
\frac{dx_\tau}{d\tau}=v(x_\tau,\tau).
$$

因此：

$$
\begin{aligned}
g'(\tau)
&=
\partial_\tau v
+
J_v\frac{dx_\tau}{d\tau}\\
&=
\partial_\tau v+J_vv.
\end{aligned}
$$

定义：

$$
\boxed{
D_\tau^vv
=
\partial_\tau v+J_vv.
}
\tag{A.20}
$$

因此：

$$
g'(\tau)=D_\tau^vv.
$$

继续求导：

$$
\boxed{
g''(\tau)=(D_\tau^v)^2v.
}
\tag{A.21}
$$

### A.5.2 在起始时间 $s$ 展开

令：

$$
\tau=s+\delta.
$$

Taylor 展开：

$$
\begin{aligned}
v(x_{s+\delta},s+\delta)
={}&
v(x_s,s)\\
&+
\delta D_s^vv\\
&+
\frac{\delta^2}{2}(D_s^v)^2v\\
&+
O(\delta^3).
\end{aligned}
\tag{A.22}
$$

平均速度：

$$
u(x_s,s,s+h)
=
\frac1h
\int_s^{s+h}
v(x_\tau,\tau)\,d\tau.
$$

令：

$$
\delta=\tau-s,
$$

得到：

$$
u
=
\frac1h
\int_0^h
v(x_{s+\delta},s+\delta)\,d\delta.
$$

代入 Taylor 展开：

$$
\begin{aligned}
u
={}&
\frac1h\int_0^h v_s\,d\delta\\
&+
\frac1h\int_0^h
\delta D_s^vv\,d\delta\\
&+
\frac1{2h}\int_0^h
\delta^2(D_s^v)^2v\,d\delta\\
&+
O(h^3).
\end{aligned}
$$

分别计算。

第一项：

$$
\frac1h\int_0^h v_s\,d\delta=v_s.
$$

第二项：

$$
\frac1h\int_0^h\delta\,d\delta
=
\frac h2.
$$

第三项：

$$
\frac1{2h}\int_0^h\delta^2\,d\delta
=
\frac{h^2}{6}.
$$

最终：

$$
\boxed{
\begin{aligned}
u(x_s,s,s+h)
={}&
v(x_s,s)\\
&+
\frac h2D_s^vv\\
&+
\frac{h^2}{6}(D_s^v)^2v\\
&+
O(h^3).
\end{aligned}
}
\tag{A.23}
$$

### A.5.3 这个公式说明什么？

平均速度包含：

$$
v,
$$

也包含：

$$
D_s^vv,
$$

以及更高阶的沿轨迹变化信息。

其中：

$$
D_s^vv
$$

可以理解为沿 ODE 轨迹的加速度。

因此：

$$
u
$$

不仅表示当前速度，还隐含了未来一段时间内速度如何变化的信息。

当：

$$
h\rightarrow0,
$$

高阶项消失：

$$
u\rightarrow v.
$$

当：

$$
h
$$

较大时，高阶运动信息变得更加重要。

这正是 MeanFlow 能够预测有限时间净位移的原因。

---

## A.6 JVP 与自动微分

### A.6.1 JVP 的数学定义

设：

$$
f:\mathbb R^n\rightarrow\mathbb R^m.
$$

在：

$$
x
$$

处的 Jacobian：

$$
J_f(x)\in\mathbb R^{m\times n}.
$$

给定方向：

$$
a\in\mathbb R^n,
$$

JVP 为：

$$
\boxed{
\operatorname{JVP}(f,x,a)
=
J_f(x)a.
}
\tag{A.24}
$$

它也可以写成方向导数：

$$
\boxed{
J_f(x)a
=
\left.
\frac{d}{dh}f(x+ha)
\right|_{h=0}.
}
\tag{A.25}
$$

### A.6.2 Forward-mode AD

对于复合函数：

$$
f=f_L\circ\cdots\circ f_1,
$$

普通前向传播：

$$
x_0\rightarrow x_1\rightarrow\cdots\rightarrow x_L.
$$

Forward-mode AD 同时传播切向量：

$$
\dot x_0=a.
$$

每一层：

$$
\boxed{
\dot x_{i+1}
=
J_{f_{i+1}}(x_i)\dot x_i.
}
\tag{A.26}
$$

最终：

$$
\boxed{
\dot x_L=J_f(x_0)a.
}
\tag{A.27}
$$

因此，不需要显式构造完整 Jacobian。

### A.6.3 JVP 与 VJP

JVP：

$$
\boxed{
J_fa.
}
\tag{A.28}
$$

VJP：

$$
\boxed{
b^\top J_f.
}
\tag{A.29}
$$

JVP 对应前向模式自动微分。

VJP 对应反向模式自动微分。

MeanFlow 需要：

$$
J_{u_\theta}
\begin{bmatrix}
V\\1\\0
\end{bmatrix}.
$$

因此 JVP 是自然的计算方式。

---

## A.7 Stop-gradient 与半梯度优化

MeanFlow 的目标依赖当前网络：

$$
u_{\mathrm{target}}(\theta)
=
V+\Delta D_su_\theta.
$$

但训练使用：

$$
\operatorname{sg}(u_{\mathrm{target}}).
$$

### A.7.1 完整梯度

设：

$$
L(\theta)=\|f_\theta-y_\theta\|^2.
$$

其中：

$$
y_\theta
$$

也依赖参数。

完整梯度：

$$
\boxed{
\nabla_\theta L
=
2(J_\theta f_\theta-J_\theta y_\theta)^\top
(f_\theta-y_\theta).
}
\tag{A.30}
$$

### A.7.2 半梯度

如果停止目标梯度：

$$
L_{\mathrm{sg}}
=
\|f_\theta-\operatorname{sg}(y_\theta)\|^2,
$$

那么：

$$
\boxed{
\nabla_\theta L_{\mathrm{sg}}
=
2J_\theta f_\theta^\top
(f_\theta-y_\theta).
}
\tag{A.31}
$$

因此：

$$
J_\theta y_\theta
$$

不再参与梯度计算。

### A.7.3 固定点解释

MeanFlow 的理想解满足：

$$
u=v+\Delta D_s^vu.
$$

定义算子：

$$
\boxed{
\mathcal T[u]
=
v+\Delta D_s^vu.
}
\tag{A.32}
$$

那么真实平均速度满足：

$$
\boxed{
u=\mathcal T[u].
}
\tag{A.33}
$$

也就是说，真实平均速度是这个算子的固定点。

MeanFlow 通过网络参数化和停止梯度回归，学习满足这一固定点关系的函数。

---

## A.8 条件速度目标的方差结构

定义：

$$
V=x_1-x_0.
$$

边缘速度：

$$
v(x,s)
=
\mathbb E[V\mid x_s=x].
$$

MeanFlow 随机训练目标：

$$
\hat u_{\mathrm{target}}
=
V+\Delta(J_{u_\theta}V+\partial_su_\theta).
$$

定义：

$$
\boxed{
A_\theta=I+\Delta J_{u_\theta}.
}
\tag{A.34}
$$

因此：

$$
\boxed{
\hat u_{\mathrm{target}}
=
A_\theta V+\Delta\partial_su_\theta.
}
\tag{A.35}
$$

固定：

$$
x_s=x,s,t,
$$

条件期望：

$$
\boxed{
\bar u_{\mathrm{target}}
=
A_\theta v+\Delta\partial_su_\theta.
}
\tag{A.36}
$$

两式相减：

$$
\boxed{
\hat u_{\mathrm{target}}
-
\bar u_{\mathrm{target}}
=
A_\theta(V-v).
}
\tag{A.37}
$$

定义条件协方差：

$$
\Sigma_V(x,s)
=
\operatorname{Cov}(V\mid x_s=x).
$$

那么：

$$
\boxed{
\operatorname{Cov}
(\hat u_{\mathrm{target}}\mid x_s=x,s,t)
=
A_\theta\Sigma_VA_\theta^\top.
}
\tag{A.38}
$$

这说明，MeanFlow 的训练目标方差不仅取决于条件速度的随机性，还取决于网络 Jacobian。

对于平方误差：

$$
\boxed{
\begin{aligned}
&\mathbb E
\left[
\|u_\theta-\hat u_{\mathrm{target}}\|^2
\mid x_s=x,s,t
\right]\\
={}&
\|u_\theta-\bar u_{\mathrm{target}}\|^2\\
&+
\operatorname{tr}
\left(
A_\theta\Sigma_VA_\theta^\top
\right).
\end{aligned}
}
\tag{A.39}
$$

这里：

$$
\operatorname{tr}
$$

表示矩阵迹。

这个结果解释了为什么条件速度虽然能够提供正确的期望监督，但训练目标仍然存在随机性。

---

## A.9 时间反演：为什么我们修改后的公式与原论文等价？

这是本次符号统一最重要的数学补充。

原论文使用：

$$
z_0=x_{\mathrm{data}},
\qquad
z_1=\epsilon.
$$

生成方向：

$$
1\rightarrow0.
$$

我们使用：

$$
x_0=\epsilon,
\qquad
x_1=x_{\mathrm{data}}.
$$

生成方向：

$$
0\rightarrow1.
$$

定义时间反演：

$$
\boxed{
x_s=z_{1-s}.
}
\tag{A.40}
$$

### A.9.1 速度如何变化？

原论文：

$$
\frac{dz_\tau}{d\tau}=v_{\mathrm{old}}(z_\tau,\tau).
$$

因为：

$$
x_s=z_{1-s},
$$

对 $s$ 求导：

$$
\frac{dx_s}{ds}
=
\frac{dz_{1-s}}{d(1-s)}
\frac{d(1-s)}{ds}.
$$

其中：

$$
\frac{d(1-s)}{ds}=-1.
$$

因此：

$$
\boxed{
v_{\mathrm{new}}(x_s,s)
=
-v_{\mathrm{old}}(z_{1-s},1-s).
}
\tag{A.41}
$$

所以速度方向必须取反。

原论文的条件速度：

$$
\epsilon-x_{\mathrm{data}}
$$

在我们的约定下变成：

$$
\boxed{
x_1-x_0.
}
\tag{A.42}
$$

### A.9.2 平均速度如何变化？

原论文平均速度：

$$
u_{\mathrm{old}}(z_\tau,r,\tau),
\qquad r\leq\tau.
$$

在新时间方向下，对应：

$$
s=1-\tau,
$$

$$
t=1-r.
$$

因此：

$$
s\leq t.
$$

由于速度反向：

$$
\boxed{
u_{\mathrm{new}}(x_s,s,t)
=
-u_{\mathrm{old}}(z_{1-s},1-t,1-s).
}
\tag{A.43}
$$

这就是完整的平均速度变换关系。

### A.9.3 MeanFlow Identity 如何变化？

原论文：

$$
u_{\mathrm{old}}
=
v_{\mathrm{old}}
-
(\tau-r)
\frac{d}{d\tau}u_{\mathrm{old}}.
$$

利用：

$$
u_{\mathrm{new}}=-u_{\mathrm{old}},
$$

$$
v_{\mathrm{new}}=-v_{\mathrm{old}},
$$

以及：

$$
\tau-r=t-s,
$$

可得到：

$$
\boxed{
u_{\mathrm{new}}
=
v_{\mathrm{new}}
+
(t-s)
\frac{d}{ds}u_{\mathrm{new}}.
}
\tag{A.44}
$$

因此，新公式中的加号并不是人为修改，而是时间反演后自然得到的结果。

### A.9.4 JVP 方向如何变化？

原论文：

$$
(v_{\mathrm{old}},0,1)
$$

对应输入：

$$
(z,r,\tau).
$$

我们采用：

$$
(x,s,t).
$$

其中：

$$
s=1-\tau,
\qquad
t=1-r.
$$

由于我们对起始时间 $s$ 求导，并保持目标时间 $t$ 不变：

$$
\frac{dx_s}{ds}=v_{\mathrm{new}},
$$

$$
\frac{ds}{ds}=1,
$$

$$
\frac{dt}{ds}=0.
$$

因此：

$$
\boxed{
\text{新 JVP 方向}=(v_{\mathrm{new}},1,0).
}
\tag{A.45}
$$

训练时使用条件速度：

$$
V=x_1-x_0,
$$

所以：

$$
\boxed{
\operatorname{JVP}
\left(
u_\theta,
(x_s,s,t),
(V,1,0)
\right).
}
\tag{A.46}
$$

至此，平均速度定义、MeanFlow Identity、训练目标、JVP 和推理公式都完成了严格的时间方向统一。



---

# 最终总结

在统一采用：

$$
\boxed{
x_0=\text{噪声},
\qquad
x_1=\text{真实数据},
\qquad
0\rightarrow1
}
$$

的约定后，MeanFlow 的完整逻辑为：

$$
\begin{gathered}
x_0\sim\mathcal N(0,I),
\qquad
x_1\sim p_{\mathrm{data}}
\\[6pt]
\Downarrow
\\[6pt]
x_s=(1-s)x_0+sx_1
\\[6pt]
\Downarrow
\\[6pt]
V=x_1-x_0
\\[6pt]
\Downarrow
\\[6pt]
v(x_s,s)=\mathbb E[V\mid x_s]
\\[6pt]
\Downarrow
\\[6pt]
\frac{dx_s}{ds}=v(x_s,s)
\\[6pt]
\Downarrow
\\[6pt]
u(x_s,s,t)
=
\frac{1}{t-s}
\int_s^t v(x_\tau,\tau)\,d\tau
\\[6pt]
\Downarrow
\\[6pt]
\boxed{
u=v+(t-s)(J_uv+\partial_su)
}
\\[6pt]
\Downarrow
\\[6pt]
\boxed{
u_{\mathrm{target}}
=
V+(t-s)
\operatorname{JVP}
\left(
u_\theta,
(x_s,s,t),
(V,1,0)
\right)
}
\\[6pt]
\Downarrow
\\[6pt]
\mathcal L_{\mathrm{MF}}
=
\mathbb E
\left[
\left\|u_\theta-
\operatorname{sg}(u_{\mathrm{target}})\right\|^2
\right]
\\[6pt]
\Downarrow
\\[6pt]
\boxed{
\hat x_1=x_0+u_\theta(x_0,0,1)
}
\end{gathered}
$$

**以后使用这套符号时，最重要的是保持三个地方一致：**

1. 条件速度为 $V=x_1-x_0$。
2. MeanFlow Identity 为 $u=v+(t-s)D_s^v u$。
3. 一步生成为 $x_1=x_0+u_\theta(x_0,0,1)$。

这样整套数学推导就与常见的 $0\rightarrow1$ Flow Matching 生成过程保持一致，也更方便后续研究与实现。

**论文来源：** [Mean Flows for One-step Generative Modeling](https://arxiv.org/abs/2505.13447) · [HTML 全文及附录](https://arxiv.org/html/2505.13447)